# CPV301 D1 checkpoint/recovery smoke R4-R3

This successor notebook uses only train/val. Run `MODE = 'fresh'` first: it intentionally pauses after one completed epoch and leaves a verified Drive checkpoint. Disconnect the runtime, reconnect a new L4 runtime, change only `MODE = 'resume'`, and Run all again. Do not start the 100-epoch pilot until the resulting package passes independent QA.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import hashlib
import re
from pathlib import Path

MODE = "fresh"  # first run: fresh; after reconnecting a new runtime: resume
CODE_COMMIT = "PENDING_AFTER_QA"  # fail-closed until R4 QA PASS and release commit
RUN_LABEL = "checkpoint_smoke_seed_20261018_r4_r3"
DRIVE_ROOT = Path("/content/drive/MyDrive/CPV301/Traffic_Sign_Detector/M1_D1_Colab")
DRIVE_BUNDLE = DRIVE_ROOT / "m1_colab_train_val_backend_r1.tar.gz"
SIDECAR = DRIVE_ROOT / "m1_colab_train_val_backend_r1.tar.gz.manifest.json"
STATE_DIR = DRIVE_ROOT / "runs" / RUN_LABEL
REPO_DIR = Path("/content/Traffic_Sign_Detector")
REPO_URL = "https://github.com/ScottPhantom/Traffic_SIgn_Detection.git"
EXPECTED_BUNDLE_BYTES = 766226800
EXPECTED_BUNDLE_SHA256 = "7ea344cb1e2c166786d0552f215993969efec9b5f06a9bf49310d4390243d641"
EXPECTED_SIDECAR_BYTES = 1499866
EXPECTED_SIDECAR_SHA256 = "7e0dc5b0144faa5f0be407224eaf8070dd51199fa8a1d2f164b248fd6a06d5b2"
PART_SPECS = (
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-00",
        99614720,
        "53f30f08406fbef5b8451610b2fd44a1f2be42ca6d50f612b68e5ace285e66ff",
    ),
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-01",
        99614720,
        "8451fe932760179d4b2e1a9cbdd21fd58744bc6b8f3d2a90e431cece7f546544",
    ),
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-02",
        99614720,
        "867c935bf152ddfc3e7919547dccf3682291326b60b956a8100fc1a59b60e98d",
    ),
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-03",
        99614720,
        "046fc8ae5d4e0a91196ccd6a6afa422fb1254baea7ae30bc3cab39602b534907",
    ),
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-04",
        99614720,
        "6d3751d858232fada5d57e8e892f3e272cd5b9e34efb184971b3f4786842665d",
    ),
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-05",
        99614720,
        "938106e4d34743e9b79109feb98d573f317afad44289f792f070f22e3924159b",
    ),
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-06",
        99614720,
        "3497c1a10314c74d4b81ba9a5a49653b16ba2e2d52b424fd7655623547cb6408",
    ),
    (
        "m1_colab_train_val_backend_r1.tar.gz.part-07",
        68923760,
        "5f5ca85bc392310b2b5693ad9d0723f046eca9fd7980b45c944f22a3b3bc0eea",
    ),
)
assert MODE in {"fresh", "resume"}
assert re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT), "Wait for QA, commit and push before Colab"


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [ ]:
assert SIDECAR.is_file() and SIDECAR.stat().st_size == EXPECTED_SIDECAR_BYTES
assert sha256_file(SIDECAR) == EXPECTED_SIDECAR_SHA256
if DRIVE_BUNDLE.is_file():
    BUNDLE = DRIVE_BUNDLE
    bundle_source = "single_archive"
else:
    parts = []
    for name, expected_bytes, expected_sha256 in PART_SPECS:
        part = DRIVE_ROOT / name
        assert part.is_file() and part.stat().st_size == expected_bytes, part
        assert sha256_file(part) == expected_sha256, part
        parts.append(part)
    BUNDLE = Path("/content/m1_colab_train_val_backend_r1.tar.gz")
    if not BUNDLE.exists():
        with BUNDLE.open("xb") as destination:
            for part in parts:
                with part.open("rb") as source:
                    for chunk in iter(lambda: source.read(1024 * 1024), b""):
                        destination.write(chunk)
    bundle_source = "verified_drive_parts"
assert BUNDLE.stat().st_size == EXPECTED_BUNDLE_BYTES
assert sha256_file(BUNDLE) == EXPECTED_BUNDLE_SHA256
if MODE == "fresh":
    assert not STATE_DIR.exists(), f"Fresh state already exists: {STATE_DIR}"
else:
    assert (STATE_DIR / "recovery/latest.json").is_file(), "Verified recovery pointer missing"
print({"mounted_bundle_preflight": "PASS", "source": bundle_source, "mode": MODE})

In [ ]:
import shutil
import subprocess
import sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", CODE_COMMIT], cwd=REPO_DIR, check=True)
observed = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True, capture_output=True, text=True
).stdout.strip()
assert observed == CODE_COMMIT
assert not subprocess.run(
    ["git", "status", "--porcelain"], cwd=REPO_DIR, check=True, capture_output=True, text=True
).stdout.strip()
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
print({"code_commit": observed, "checkout_clean": True})

In [ ]:
command = [
    sys.executable,
    str(REPO_DIR / "detector/colab/run_d1_colab_checkpoint_r3.py"),
    "--repo-root",
    str(REPO_DIR),
    "--bundle",
    str(BUNDLE),
    "--state-dir",
    str(STATE_DIR),
    "--mode",
    MODE,
    "--seed",
    "20261018",
    "--epochs",
    "2",
    "--imgsz",
    "1280",
    "--batch",
    "4",
    "--device",
    "0",
    "--tag",
    "checkpoint_smoke_r4_r3",
    "--download-approved-weight",
]
if MODE == "fresh":
    command += ["--stop-after-epoch", "1"]
print("Running:", " ".join(command))
subprocess.run(command, check=True)

In [ ]:
import json

execution = json.loads((STATE_DIR / "colab_execution.json").read_text())
latest = json.loads((STATE_DIR / "recovery/latest.json").read_text())
assert execution["sealed_test_accessed"] is False
if MODE == "fresh":
    assert execution["status"] == "PAUSED_FOR_RESUME", execution
    assert latest["generation"] == "epoch-0001", latest
    print("Checkpoint smoke leg PASS. Disconnect runtime, reconnect L4, set MODE=resume, Run all.")
else:
    assert execution["status"] == "SUCCEEDED", execution
    archives = sorted(STATE_DIR.glob("*_result.tar.gz"))
    assert len(archives) == 1, archives
    print({"resume_smoke": "PASS", "archive": str(archives[0]), "latest": latest})
    import importlib.util
    import sys

    verifier_path = REPO_DIR / "detector/colab/run_d1_colab_checkpoint_r3.py"
    spec = importlib.util.spec_from_file_location("d1_r4_live_verifier", verifier_path)
    assert spec is not None and spec.loader is not None
    verifier = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = verifier
    spec.loader.exec_module(verifier)
    runtime_evidence = verifier.verify_live_state(STATE_DIR)
    assert runtime_evidence["runtime_evidence"]["distinct_runtime_digests"] is True
    assert runtime_evidence["runtime_evidence"]["sealed_test_accessed"] is False
    print(runtime_evidence)
